**CI twin of `ch01-what-is-learning.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv

homes = load_csv("california-housing-sample")
print(homes.shape)
print(homes[["MedInc", "HouseAge", "AveRooms", "MedHouseVal"]].head(3))

In [ ]:
prices = homes["MedHouseVal"]

# Rule A: every California house costs about $300k.
guess_a = 3.0
# Rule B: price tracks income one-for-one.
guess_b = homes["MedInc"]
# Rule C: refined — half the income figure, plus a base of $100k.
guess_c = homes["MedInc"] / 2 + 1

for name, guess in [("A: flat $300k", guess_a),
                    ("B: price = income", guess_b),
                    ("C: income/2 + 1", guess_c)]:
    mae = (prices - guess).abs().mean()
    print(f"Rule {name:18} average miss: {mae:.3f}")

In [ ]:
learned_mean = prices.mean()
mae_mean = (prices - learned_mean).abs().mean()

print(f"learned rule: always predict {learned_mean:.3f}")
print(f"average miss: {mae_mean:.3f}")

In [ ]:
import pandas as pd

buckets = pd.cut(homes["MedInc"], bins=[0, 2.5, 4.5, 100],
                 labels=["low", "mid", "high"])
group_price = prices.groupby(buckets, observed=True).mean()
print(group_price.round(3))

predictions = buckets.map(group_price).astype(float)
mae_buckets = (prices - predictions).abs().mean()
print(f"average miss: {mae_buckets:.3f}")

In [ ]:
def price_by_income(med_inc):
    return med_inc / 2 + 1

run_tests([
    ("income 3.0", price_by_income(3.0), 2.5),
    ("income 5.0", price_by_income(5.0), 3.5),
    ("income 0.0", price_by_income(0.0), 1.0),
])

In [ ]:
def mae(y_true, y_pred):
    gaps = [abs(t - p) for t, p in zip(y_true, y_pred)]
    return sum(gaps) / len(gaps)

run_tests([
    ("mixed gaps", mae([2.0, 3.0, 1.0], [1.5, 3.5, 2.0]), 2 / 3),
    ("perfect predictions", mae([1.0, 1.0], [1.0, 1.0]), 0.0),
    ("symmetric misses", mae([0.0, 4.0], [2.0, 2.0]), 2.0),
], tol=1e-9)